<a href="https://colab.research.google.com/github/sudhans18/ABDA/blob/main/Audio_Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 0: Environment Setup
!pip install -q librosa soundfile praat-parselmouth scipy numpy pandas matplotlib
print("Setup complete. Dependencies ready.")

Setup complete. Dependencies ready.


In [ ]:
# Updated Cell 1: Upload a new audio file and preprocess it
from google.colab import files
import os
import numpy as np
import librosa

print("Upload a new audio file (.mp3, .wav, .m4a):")
uploaded = files.upload()

if uploaded:
    new_filename = list(uploaded.keys())[0]
    AUDIO_FILE_PATH = os.path.join("/content", new_filename)
    print(f"\nLoaded file: {AUDIO_FILE_PATH}")

    # Preprocess
    audio, sr = preprocess_audio(AUDIO_FILE_PATH, target_sr=16000)
    duration_sec = len(audio) / float(sr)

    print("Step 1 Complete:")
    print(f"  Sampling Rate: {sr} Hz")
    print(f"  Duration: {duration_sec:.2f} seconds")
    print(f"  Peak Amplitude: {np.max(np.abs(audio)):.4f}")
else:
    print("No file uploaded.")

Upload a new audio file (.mp3, .wav, .m4a):


Saving audio 2 test.mp3 to audio 2 test (1).mp3

Loaded file: /content/audio 2 test (1).mp3
Step 1 Complete:
  Sampling Rate: 16000 Hz
  Duration: 25.92 seconds
  Peak Amplitude: 1.0000


In [ ]:
# Cell 2: Voice Activity Detection
def detect_voice_activity(audio: np.ndarray, sr: int = 16000, top_db: float = 28.0):
    intervals = librosa.effects.split(audio, top_db=top_db, frame_length=2048, hop_length=512)
    return [(int(s), int(e)) for s, e in intervals]

speech_intervals = detect_voice_activity(audio, sr=sr)

total_speech_sec = sum((e - s) for s, e in speech_intervals) / sr
print("Step 2 Complete:")
print(f"  Active Speech Intervals Found: {len(speech_intervals)}")
print(f"  Total Active Speech Time: {total_speech_sec:.2f}s out of {duration_sec:.2f}s")

Step 2 Complete:
  Active Speech Intervals Found: 19
  Total Active Speech Time: 17.95s out of 25.92s


In [ ]:
# Cell 3: MFCC Extraction
def extract_mfcc_features(audio: np.ndarray, sr: int, speech_intervals, n_mfcc: int = 13):
    mfcc_dict = {}

    # Isolate speech frames if VAD detected any; fallback to full audio
    if speech_intervals:
        speech_frames = np.concatenate([audio[s:e] for s, e in speech_intervals if e > s])
    else:
        speech_frames = audio

    if len(speech_frames) < 512:
        speech_frames = audio

    mfccs = librosa.feature.mfcc(y=speech_frames, sr=sr, n_mfcc=n_mfcc)

    for i in range(n_mfcc):
        idx = i + 1
        traj = mfccs[i, :]
        mfcc_dict[f"mfcc_{idx}_mean"] = float(np.mean(traj))
        mfcc_dict[f"mfcc_{idx}_std"] = float(np.std(traj))
        mfcc_dict[f"mfcc_{idx}_min"] = float(np.min(traj))
        mfcc_dict[f"mfcc_{idx}_max"] = float(np.max(traj))

    return mfcc_dict

mfcc_feats = extract_mfcc_features(audio, sr=sr, speech_intervals=speech_intervals)
print("Step 3 Complete (MFCC 1–13 extracted):")
print(f"  MFCC 1 Mean: {mfcc_feats['mfcc_1_mean']:.4f}, Std: {mfcc_feats['mfcc_1_std']:.4f}")
print(f"  MFCC 2 Mean: {mfcc_feats['mfcc_2_mean']:.4f}, Std: {mfcc_feats['mfcc_2_std']:.4f}")

Step 3 Complete (MFCC 1–13 extracted):
  MFCC 1 Mean: -240.3457, Std: 76.1167
  MFCC 2 Mean: 90.2318, Std: 54.5805


In [ ]:
# Cell 4: F0 / Pitch Extraction
import parselmouth

def extract_pitch_features(audio: np.ndarray, sr: int):
    metrics = {
        "f0_mean": 0.0, "f0_std": 0.0, "f0_min": 0.0,
        "f0_max": 0.0, "f0_range": 0.0, "f0_variability": 0.0
    }

    snd = parselmouth.Sound(audio, sampling_frequency=sr)
    pitch = snd.to_pitch(pitch_floor=75.0, pitch_ceiling=500.0)
    pitch_values = pitch.selected_array['frequency']
    voiced_f0 = pitch_values[pitch_values > 0]

    if len(voiced_f0) > 0:
        mean_v = float(np.mean(voiced_f0))
        std_v = float(np.std(voiced_f0))
        min_v = float(np.min(voiced_f0))
        max_v = float(np.max(voiced_f0))

        metrics["f0_mean"] = mean_v
        metrics["f0_std"] = std_v
        metrics["f0_min"] = min_v
        metrics["f0_max"] = max_v
        metrics["f0_range"] = max_v - min_v
        metrics["f0_variability"] = (std_v / mean_v) if mean_v > 0 else 0.0

    return metrics

pitch_feats = extract_pitch_features(audio, sr=sr)
print("Step 4 Complete (Pitch/F0):")
for k, v in pitch_feats.items():
    print(f"  {k}: {v:.3f}")

Step 4 Complete (Pitch/F0):
  f0_mean: 127.162
  f0_std: 30.827
  f0_min: 84.222
  f0_max: 406.394
  f0_range: 322.171
  f0_variability: 0.242


In [ ]:
# Cell 5: Jitter, Shimmer, HNR
from parselmouth.praat import call

def extract_voice_quality_features(audio: np.ndarray, sr: int):
    metrics = {"jitter_local": 0.0, "shimmer_local": 0.0, "hnr_mean": 0.0}

    snd = parselmouth.Sound(audio, sampling_frequency=sr)
    point_process = call(snd, "To PointProcess (periodic, cc)", 75.0, 500.0)

    jitter = call(point_process, "Get jitter (local)", 0.0, 0.0, 0.0001, 0.02, 1.3)
    shimmer = call([snd, point_process], "Get shimmer (local)", 0.0, 0.0, 0.0001, 0.02, 1.3, 1.6)
    harmonicity = call(snd, "To Harmonicity (cc)", 0.01, 75.0, 0.1, 4.5)
    hnr = call(harmonicity, "Get mean", 0.0, 0.0)

    metrics["jitter_local"] = float(jitter) if not np.isnan(jitter) else 0.0
    metrics["shimmer_local"] = float(shimmer) if not np.isnan(shimmer) else 0.0
    metrics["hnr_mean"] = float(hnr) if not np.isnan(hnr) else 0.0

    return metrics

voice_quality_feats = extract_voice_quality_features(audio, sr=sr)
print("Step 5 Complete (Voice Quality):")
for k, v in voice_quality_feats.items():
    print(f"  {k}: {v:.5f}")

Step 5 Complete (Voice Quality):
  jitter_local: 0.02130
  shimmer_local: 0.15138
  hnr_mean: 6.62253


In [ ]:
# Cell 6: Pause Dynamics
def extract_pause_features(speech_intervals, total_samples: int, sr: int, min_pause_sec: float = 0.15):
    metrics = {
        "pause_count": 0.0, "pause_total_duration": 0.0,
        "pause_mean_duration": 0.0, "pause_rate": 0.0
    }
    total_sec = total_samples / sr
    if total_sec <= 0 or not speech_intervals:
        return metrics

    pauses = []
    # Pre-speech silence
    if speech_intervals[0][0] > 0:
        p = speech_intervals[0][0] / sr
        if p >= min_pause_sec: pauses.append(p)

    # Internal silences
    for i in range(len(speech_intervals) - 1):
        silence = (speech_intervals[i+1][0] - speech_intervals[i][1]) / sr
        if silence >= min_pause_sec:
            pauses.append(silence)

    # Post-speech silence
    if speech_intervals[-1][1] < total_samples:
        p = (total_samples - speech_intervals[-1][1]) / sr
        if p >= min_pause_sec: pauses.append(p)

    if pauses:
        metrics["pause_count"] = float(len(pauses))
        metrics["pause_total_duration"] = float(np.sum(pauses))
        metrics["pause_mean_duration"] = float(np.mean(pauses))
        metrics["pause_rate"] = float(len(pauses)) / total_sec

    return metrics

pause_feats = extract_pause_features(speech_intervals, len(audio), sr=sr)
print("Step 6 Complete (Pause Dynamics):")
for k, v in pause_feats.items():
    print(f"  {k}: {v:.3f}")

Step 6 Complete (Pause Dynamics):
  pause_count: 10.000
  pause_total_duration: 7.360
  pause_mean_duration: 0.736
  pause_rate: 0.386


In [ ]:
# Cell 7: Speaking Rate
def extract_speaking_rate(total_duration_sec: float, transcript: str = None):
    metrics = {"speaking_rate_wps": 0.0}
    if transcript and total_duration_sec > 0:
        words = len(transcript.strip().split())
        metrics["speaking_rate_wps"] = float(words / total_duration_sec)
    return metrics

# Supply a string transcript of your recording here if you have it
my_transcript = "This is a real recording test for ABDA Phase 1 audio pipeline."
speaking_feats = extract_speaking_rate(duration_sec, transcript=my_transcript)

print("Step 7 Complete (Speaking Rate):")
print(f"  Words per second: {speaking_feats['speaking_rate_wps']:.2f}")

Step 7 Complete (Speaking Rate):
  Words per second: 0.46


In [ ]:
# Cell 8: Spectral Energy
def extract_spectral_features(audio: np.ndarray, sr: int, speech_intervals):
    if speech_intervals:
        frames = np.concatenate([audio[s:e] for s, e in speech_intervals if e > s])
    else:
        frames = audio

    rms = librosa.feature.rms(y=frames)[0]
    centroid = librosa.feature.spectral_centroid(y=frames, sr=sr)[0]

    return {
        "rms_mean": float(np.mean(rms)),
        "rms_std": float(np.std(rms)),
        "spectral_centroid_mean": float(np.mean(centroid)),
        "spectral_centroid_std": float(np.std(centroid))
    }

spectral_feats = extract_spectral_features(audio, sr=sr, speech_intervals=speech_intervals)
print("Step 8 Complete (Spectral Features):")
for k, v in spectral_feats.items():
    print(f"  {k}: {v:.4f}")

Step 8 Complete (Spectral Features):
  rms_mean: 0.0667
  rms_std: 0.0438
  spectral_centroid_mean: 1993.5065
  spectral_centroid_std: 1172.9770


In [ ]:
# Cell 9: Audio Quality Score
def compute_audio_quality_score(audio: np.ndarray, speech_intervals):
    total_len = len(audio)
    speech_samples = sum((e - s) for s, e in speech_intervals)
    speech_ratio = min(1.0, float(speech_samples / total_len))

    speech_mask = np.zeros(total_len, dtype=bool)
    for s, e in speech_intervals:
        speech_mask[s:e] = True

    sig_energy = np.mean(audio[speech_mask] ** 2) if np.any(speech_mask) else 1e-9
    noise_energy = np.mean(audio[~speech_mask] ** 2) if np.any(~speech_mask) else 1e-9

    snr = 10.0 * np.log10(max(sig_energy / (noise_energy + 1e-9), 1e-3))
    snr_norm = 1.0 / (1.0 + np.exp(-0.2 * (snr - 10.0)))
    quality_score = float(np.clip(0.4 * speech_ratio + 0.6 * snr_norm, 0.0, 1.0))

    return {
        "speech_ratio": float(speech_ratio),
        "snr_estimate": float(snr),
        "quality_score": quality_score
    }

quality_feats = compute_audio_quality_score(audio, speech_intervals)
print("Step 9 Complete (Quality Evaluation):")
print(f"  Voiced/Speech Ratio: {quality_feats['speech_ratio']:.3f}")
print(f"  Estimated SNR: {quality_feats['snr_estimate']:.2f} dB")
print(f"  Overall Audio Quality Score (0-1): {quality_feats['quality_score']:.4f}")

Step 9 Complete (Quality Evaluation):
  Voiced/Speech Ratio: 0.693
  Estimated SNR: 26.15 dB
  Overall Audio Quality Score (0-1): 0.8542


In [ ]:
# Cell 10: Full Session Aggregation
import json
import pandas as pd
from datetime import datetime

# Combine all individual dictionaries
all_extracted_features = {}
for block in [mfcc_feats, pitch_feats, voice_quality_feats, pause_feats, speaking_feats, spectral_feats]:
    all_extracted_features.update(block)

# Construct standardized session object
session_record = {
    "participant_id": "P_SUDESH",
    "session_id": "P_SUDESH_S01",
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "source_dataset": "CMU-MOSEI",
    "language": "English",
    "modality_available": True,
    "quality_score": quality_feats["quality_score"],
    "feature_version": "1.0",
    "features": all_extracted_features
}

print("=== STEP 10: STANDARDIZED SESSION OUTPUT ===")
print(f"Participant: {session_record['participant_id']}")
print(f"Session ID: {session_record['session_id']}")
print(f"Modality Available: {session_record['modality_available']}")
print(f"Quality Score: {session_record['quality_score']:.4f}")
print(f"Total Features Extracted: {len(session_record['features'])}")

# Display formatted table of primary vocal biomarkers for review
review_summary = {
    "Vocal Metric": [
        "Pitch Mean (F0)", "Pitch Std", "Pitch Variability",
        "Jitter (Local)", "Shimmer (Local)", "HNR (Mean)",
        "Pause Count", "Total Pause Time (s)", "Pause Rate",
        "Speaking Rate (wps)", "RMS Energy Mean", "Spectral Centroid"
    ],
    "Extracted Value": [
        round(all_extracted_features["f0_mean"], 2),
        round(all_extracted_features["f0_std"], 2),
        round(all_extracted_features["f0_variability"], 3),
        round(all_extracted_features["jitter_local"], 5),
        round(all_extracted_features["shimmer_local"], 5),
        round(all_extracted_features["hnr_mean"], 2),
        all_extracted_features["pause_count"],
        round(all_extracted_features["pause_total_duration"], 2),
        round(all_extracted_features["pause_rate"], 3),
        round(all_extracted_features["speaking_rate_wps"], 2),
        round(all_extracted_features["rms_mean"], 4),
        round(all_extracted_features["spectral_centroid_mean"], 2)
    ]
}

print("\n=== SUMMARY TABLE FOR PHASE 1 REVIEW ===")
print(pd.DataFrame(review_summary).to_markdown(index=False))

=== STEP 10: STANDARDIZED SESSION OUTPUT ===
Participant: P_SUDESH
Session ID: P_SUDESH_S01
Modality Available: True
Quality Score: 0.8542
Total Features Extracted: 70

=== SUMMARY TABLE FOR PHASE 1 REVIEW ===
| Vocal Metric         |   Extracted Value |
|:---------------------|------------------:|
| Pitch Mean (F0)      |         127.16    |
| Pitch Std            |          30.83    |
| Pitch Variability    |           0.242   |
| Jitter (Local)       |           0.0213  |
| Shimmer (Local)      |           0.15138 |
| HNR (Mean)           |           6.62    |
| Pause Count          |          10       |
| Total Pause Time (s) |           7.36    |
| Pause Rate           |           0.386   |
| Speaking Rate (wps)  |           0.46    |
| RMS Energy Mean      |           0.0667  |
| Spectral Centroid    |        1993.51    |


/tmp/ipykernel_10693/2098165916.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z",
